In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit",
    dtype = None,
    max_seq_length = 1024, 
    load_in_4bit = True,  
    full_finetuning = False,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Preparing Model for FT

In [3]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 256,           # Larger = higher accuracy, but might overfit
    lora_alpha = 256,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing="unsloth"
)

## Format

[{'content': 'text',
  'role': 'user'},
 {'content': 'text",
  'role': 'assistant'}]

In [4]:
from datasets import Dataset
import json 

ds = "training_set.json"
with open(ds, "r", encoding="utf-8") as f:
    data = [(item[0], item[1]) for item in json.load(f)]
print(data[0])

dataset = Dataset.from_dict({
    "question": [item[0] for item in data],
    "answer": [item[1] for item in data]
})
print(dataset[0])

# Define transformation function
def convert_format(example):
    example["conversations"] = [{'content': example['question'], 'role': 'user'}, {'content': example['answer'], 'role': 'assistant'}]
    return example

dataset = dataset.map(convert_format)
print(dataset[0])


("Hi, I'm looking for guidance on building a portfolio. I'm primarily interested in dividend-paying stocks in the Utilities and Real Estate sectors. Ideally, I'd like to focus on companies with a dividend yield above 11.2%, and a price-to-earnings (P/E) ratio below 30. Could you suggest some potential investment options based on those criteria?", "Sure, let's build an optimized stock portfolio around your objectives.\nThe portfolio will contain stocks with the following qualities:\n\tBelonging to the following sectors: Utilities, Real Estate\n\tDividend rate above 11.2%\n\tPE ratio below 30\n**Disclaimer:** AI-generated information is provided for educational purposes only and should not be considered financial advice.")
{'question': "Hi, I'm looking for guidance on building a portfolio. I'm primarily interested in dividend-paying stocks in the Utilities and Real Estate sectors. Ideally, I'd like to focus on companies with a dividend yield above 11.2%, and a price-to-earnings (P/E) rat

Map:   0%|          | 0/2080 [00:00<?, ? examples/s]

{'question': "Hi, I'm looking for guidance on building a portfolio. I'm primarily interested in dividend-paying stocks in the Utilities and Real Estate sectors. Ideally, I'd like to focus on companies with a dividend yield above 11.2%, and a price-to-earnings (P/E) ratio below 30. Could you suggest some potential investment options based on those criteria?", 'answer': "Sure, let's build an optimized stock portfolio around your objectives.\nThe portfolio will contain stocks with the following qualities:\n\tBelonging to the following sectors: Utilities, Real Estate\n\tDividend rate above 11.2%\n\tPE ratio below 30\n**Disclaimer:** AI-generated information is provided for educational purposes only and should not be considered financial advice.", 'conversations': [{'content': "Hi, I'm looking for guidance on building a portfolio. I'm primarily interested in dividend-paying stocks in the Utilities and Real Estate sectors. Ideally, I'd like to focus on companies with a dividend yield above 1

In [5]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/2080 [00:00<?, ? examples/s]

In [6]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/2080 [00:00<?, ? examples/s]

In [7]:
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Hi, I'm looking for guidance on building a portfolio. I'm primarily interested in dividend-paying stocks in the Utilities and Real Estate sectors. Ideally, I'd like to focus on companies with a dividend yield above 11.2%, and a price-to-earnings (P/E) ratio below 30. Could you suggest some potential investment options based on those criteria?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Sure, let's build an optimized stock portfolio around your objectives.
The portfolio will contain stocks with the following qualities:
	Belonging to the following sectors: Utilities, Real Estate
	Dividend rate above 11.2%
	PE ratio below 30
**Disclaimer:** AI-generated information is provided for educational purposes only and should not be considered financial advice.<|eot_id|>


In [8]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, 
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 70,
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2080 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<|start_header_id|>user<|end_header_id|>\n\n' and response_part = '<|start_header_id|>assistant<|end_header_id|>\n\n'


Map:   0%|          | 0/2080 [00:00<?, ? examples/s]

In [10]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHi, Generate advice on constructing a portfolio using these tickers: BBY, CB, ALL, REGN, SLG, AKAM, and ITW. My primary goal is to minimize the risk of significant losses – essentially, I want to protect against downside risk.  I'm also concerned about keeping the investments relatively close to each other in terms of performance; I'd like to limit the dispersion of returns within the portfolio. Could you suggest an optimal allocation strategy for these stocks, keeping both downside protection and low dispersion in mind?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSure, I can put together an optimized stock portfolio for you.\nThe portfolio will consist of the following stock tickers: BBY, CB, ALL, REGN, SLG, AKAM, ITW\nPortfolio will be optimized using a dispersion-based risk measure.\n**D

In [11]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                               Sure, I can put together an optimized stock portfolio for you.\nThe portfolio will consist of the following stock tickers: BBY, CB, ALL, REGN, SLG, AKAM, ITW\nPortfolio will be optimized using a dispersion-based risk measure.\n**Disclaimer:** AI-generated information is provided for educational purposes only and should not be considered financial advice.<|eot_id|>'

In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.562 GB.
3.672 GB of memory reserved.


In [13]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,080 | Num Epochs = 1 | Total steps = 70
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 389,021,696 of 3,601,771,520 (10.80% trained)


Step,Training Loss
1,2.606700
2,2.682300
3,2.425800
4,2.032500
5,2.156900
6,1.861900
7,1.714100
8,1.876900
9,1.230900
10,1.137100


In [14]:
model.save_pretrained("./models/llamaFT")
tokenizer.save_pretrained("./models/llamaFT")

('./models/llamaFT/tokenizer_config.json',
 './models/llamaFT/special_tokens_map.json',
 './models/llamaFT/chat_template.jinja',
 './models/llamaFT/tokenizer.json')

In [15]:
import shutil

shutil.make_archive(
    "./models/llamaFT",
    "zip",
    "./models/llamaFT"
)

'/kaggle/working/models/llamaFT.zip'